# 🛒 Voice-Enabled AI Retail Assistant
### Using Amazon Rekognition + Polly + Lambda + S3

**Lab Goal:** Upload an image → Detect objects using Rekognition → Generate MP3 audio using Polly → Save to S3

---
### ✅ Pre-Requisites:
- Run this notebook on **SageMaker Jupyter** or **SageMaker Studio**
- SageMaker execution role must have **Admin / required permissions**
- `retail-shelf.jpg` image must be uploaded to the same folder as this notebook

---
### 📌 Architecture:
```
Upload Image → S3 Input Bucket → Lambda (auto-trigger) → Rekognition → Polly → S3 Output Bucket (MP3)
```
In this notebook we will run the entire pipeline directly using **boto3** (you can also test without the Lambda trigger).

---
## 🔧 STEP 0: Install & Import Required Libraries

In [1]:
# Install if missing
!pip install boto3 --quiet

import boto3
import json
import zipfile
import os
import time
import urllib.parse

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


---
## ⚙️ STEP 1: Configuration — Fill In Your Details Here

> **⚠️ Important:** Update `ACCOUNT_ID` and `REGION` with your actual AWS values below.

In [2]:
# =============================================
# 👇 ENTER YOUR DETAILS HERE
# =============================================

REGION     = 'us-east-1'           # Your AWS region
ACCOUNT_ID = '745404749079'     # e.g. '634602958025'  ← Change this
IMAGE_FILE = 'retail-shelf.jpg'    # Your local image file name

# =============================================
# Auto-generated names (do not change)
# =============================================
INPUT_BUCKET  = f'rekognition-input-images-{ACCOUNT_ID}-{REGION}-nb'
OUTPUT_BUCKET = f'rekognition-output-audio-{ACCOUNT_ID}-{REGION}-nb'
LAMBDA_ROLE   = 'SmartRetailLambdaRole'
LAMBDA_FUNC   = 'SmartRetailVisionNarrator'

print(f"📦 Input Bucket  : {INPUT_BUCKET}")
print(f"🔊 Output Bucket : {OUTPUT_BUCKET}")
print(f"🌍 Region        : {REGION}")
print(f"🔑 Account ID    : {ACCOUNT_ID}")

📦 Input Bucket  : rekognition-input-images-745404749079-us-east-1-nb
🔊 Output Bucket : rekognition-output-audio-745404749079-us-east-1-nb
🌍 Region        : us-east-1
🔑 Account ID    : 745404749079


---
## 🪣 STEP 2: Create S3 Buckets (Input + Output)

In [3]:
s3 = boto3.client('s3', region_name=REGION)

def create_bucket(bucket_name):
    try:
        if REGION == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={'LocationConstraint': REGION}
            )
        print(f"✅ Bucket created: {bucket_name}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"ℹ️  Bucket already exists (yours): {bucket_name}")
    except Exception as e:
        print(f"❌ Error: {e}")

create_bucket(INPUT_BUCKET)
create_bucket(OUTPUT_BUCKET)

✅ Bucket created: rekognition-input-images-745404749079-us-east-1-nb
✅ Bucket created: rekognition-output-audio-745404749079-us-east-1-nb


---
## 🔐 STEP 3: Create IAM Role for Lambda

> **⚠️ Note:** If you get an `AccessDenied` error here, your SageMaker execution role does not have IAM permissions.
> 
> **Fix Option A (Recommended):** Create the role manually in the AWS Console → IAM → Roles → Create Role → Select Lambda → Attach the 4 policies below → Name it `SmartRetailLambdaRole`. Then skip to Step 4 and set `LAMBDA_ROLE_ARN` manually.
>
> **Fix Option B:** Go to IAM → Roles → Find your SageMaker execution role → Attach `IAMFullAccess` and `AWSLambda_FullAccess` policies → Re-run this cell.

In [4]:
iam = boto3.client('iam', region_name=REGION)

trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
})

# Create the IAM Role
try:
    role_response = iam.create_role(
        RoleName=LAMBDA_ROLE,
        AssumeRolePolicyDocument=trust_policy,
        Description='Role for SmartRetailVisionNarrator Lambda'
    )
    LAMBDA_ROLE_ARN = role_response['Role']['Arn']
    print(f"✅ IAM Role created: {LAMBDA_ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    LAMBDA_ROLE_ARN = iam.get_role(RoleName=LAMBDA_ROLE)['Role']['Arn']
    print(f"ℹ️  Role already exists: {LAMBDA_ROLE_ARN}")

# Attach 4 required policies
policies = [
    'arn:aws:iam::aws:policy/AmazonRekognitionFullAccess',
    'arn:aws:iam::aws:policy/AmazonPollyFullAccess',
    'arn:aws:iam::aws:policy/AmazonS3FullAccess',
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
]

for policy in policies:
    iam.attach_role_policy(RoleName=LAMBDA_ROLE, PolicyArn=policy)
    print(f"   ✅ Attached: {policy.split('/')[-1]}")

print("\n⏳ Waiting 10 seconds for the role to propagate...")
time.sleep(10)
print("✅ IAM setup complete!")

✅ IAM Role created: arn:aws:iam::745404749079:role/SmartRetailLambdaRole
   ✅ Attached: AmazonRekognitionFullAccess
   ✅ Attached: AmazonPollyFullAccess


   ✅ Attached: AmazonS3FullAccess
   ✅ Attached: AWSLambdaBasicExecutionRole

⏳ Waiting 10 seconds for the role to propagate...


✅ IAM setup complete!


In [ ]:
# ============================================================
# IF STEP 3 FAILED with AccessDenied → Run this cell instead
# Paste the ARN of the role you created manually in Console
# ============================================================

# LAMBDA_ROLE_ARN = 'arn:aws:iam::745404749079:role/SmartRetailLambdaRole'  # ← Paste your ARN here
# print(f"✅ Role ARN set manually: {LAMBDA_ROLE_ARN}")

---
## 🪣 STEP 4: Set S3 Bucket Policies

In [5]:
# Input bucket policy
input_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowRekognitionService",
            "Effect": "Allow",
            "Principal": {"Service": "rekognition.amazonaws.com"},
            "Action": ["s3:GetObject"],
            "Resource": f"arn:aws:s3:::{INPUT_BUCKET}/*"
        },
        {
            "Sid": "AllowLambdaRole",
            "Effect": "Allow",
            "Principal": {"AWS": LAMBDA_ROLE_ARN},
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{INPUT_BUCKET}",
                f"arn:aws:s3:::{INPUT_BUCKET}/*"
            ]
        }
    ]
})

# Output bucket policy
output_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "AllowLambdaPutObject",
        "Effect": "Allow",
        "Principal": {"AWS": LAMBDA_ROLE_ARN},
        "Action": ["s3:PutObject", "s3:GetObject", "s3:ListBucket"],
        "Resource": [
            f"arn:aws:s3:::{OUTPUT_BUCKET}",
            f"arn:aws:s3:::{OUTPUT_BUCKET}/*"
        ]
    }]
})

s3.put_bucket_policy(Bucket=INPUT_BUCKET,  Policy=input_policy)
print(f"✅ Input bucket policy applied  : {INPUT_BUCKET}")

s3.put_bucket_policy(Bucket=OUTPUT_BUCKET, Policy=output_policy)
print(f"✅ Output bucket policy applied : {OUTPUT_BUCKET}")

✅ Input bucket policy applied  : rekognition-input-images-745404749079-us-east-1-nb
✅ Output bucket policy applied : rekognition-output-audio-745404749079-us-east-1-nb


---
## ⚡ STEP 5: Create and Deploy the Lambda Function

> **⚠️ Note:** If you get an `AccessDenied` error here, attach `AWSLambda_FullAccess` to your SageMaker execution role in IAM Console, then re-run.

In [6]:
# Lambda Python code
lambda_code = f'''
import boto3
import json
import urllib.parse

rekognition = boto3.client('rekognition', region_name='{REGION}')
polly       = boto3.client('polly',       region_name='{REGION}')
s3          = boto3.client('s3',          region_name='{REGION}')

OUTPUT_BUCKET = '{OUTPUT_BUCKET}'

def lambda_handler(event, context):
    bucket    = event['Records'][0]['s3']['bucket']['name']
    image_key = urllib.parse.unquote_plus(event['Records'][0]['s3']['object']['key'])

    # Step 1: Detect labels using Rekognition
    response = rekognition.detect_labels(
        Image={{"S3Object": {{"Bucket": bucket, "Name": image_key}}}},
        MaxLabels=5,
        MinConfidence=80
    )

    # Step 2: Convert labels into a spoken sentence
    labels = response['Labels']
    if not labels:
        description = "I could not identify any objects in this image."
    else:
        description = "I can see the following items. "
        for label in labels:
            description += f"{{label['Name']}}, "

    # Step 3: Generate MP3 audio using Polly
    audio = polly.synthesize_speech(
        Text=description,
        OutputFormat='mp3',
        VoiceId='Joanna'
    )

    # Step 4: Save the MP3 to the output S3 bucket
    audio_key = image_key.replace('.jpg', '.mp3')
    s3.put_object(
        Bucket=OUTPUT_BUCKET,
        Key=audio_key,
        Body=audio['AudioStream'].read(),
        ContentType='audio/mpeg'
    )

    return {{"statusCode": 200, "body": json.dumps(f"Audio saved: {{audio_key}}") }}
'''

# Write the Lambda source file
with open('/tmp/lambda_function.py', 'w') as f:
    f.write(lambda_code)

# Package it into a ZIP file
with zipfile.ZipFile('/tmp/lambda_function.zip', 'w') as z:
    z.write('/tmp/lambda_function.py', 'lambda_function.py')

print("✅ Lambda source file and ZIP package are ready!")

✅ Lambda source file and ZIP package are ready!


In [7]:
lmb = boto3.client('lambda', region_name=REGION)

with open('/tmp/lambda_function.zip', 'rb') as f:
    zip_bytes = f.read()

try:
    response = lmb.create_function(
        FunctionName=LAMBDA_FUNC,
        Runtime='python3.10',
        Role=LAMBDA_ROLE_ARN,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': zip_bytes},
        Timeout=60,
        MemorySize=256,
        Description='Smart Retail Vision Narrator - Rekognition + Polly pipeline'
    )
    LAMBDA_ARN = response['FunctionArn']
    print(f"✅ Lambda function created: {LAMBDA_ARN}")
except lmb.exceptions.ResourceConflictException:
    # Function already exists, update the code
    response = lmb.update_function_code(
        FunctionName=LAMBDA_FUNC,
        ZipFile=zip_bytes
    )
    LAMBDA_ARN = response['FunctionArn']
    print(f"ℹ️  Lambda already exists, code updated: {LAMBDA_ARN}")

print("⏳ Waiting 5 seconds for Lambda to finish deploying...")
time.sleep(5)
print("✅ Lambda deployed successfully!")

✅ Lambda function created: arn:aws:lambda:us-east-1:745404749079:function:SmartRetailVisionNarrator
⏳ Waiting 5 seconds for Lambda to finish deploying...


✅ Lambda deployed successfully!


---
## 🔔 STEP 6: Add S3 Trigger to Lambda
> Every time a `.jpg` file is uploaded to the input bucket, Lambda will fire automatically.

In [8]:
# Grant S3 permission to invoke the Lambda function
try:
    lmb.add_permission(
        FunctionName=LAMBDA_FUNC,
        StatementId='S3InvokePermission',
        Action='lambda:InvokeFunction',
        Principal='s3.amazonaws.com',
        SourceArn=f'arn:aws:s3:::{INPUT_BUCKET}',
        SourceAccount=ACCOUNT_ID
    )
    print("✅ Lambda invoke permission granted to S3")
except lmb.exceptions.ResourceConflictException:
    print("ℹ️  Permission already exists")

# Configure S3 bucket notification to trigger Lambda
s3.put_bucket_notification_configuration(
    Bucket=INPUT_BUCKET,
    NotificationConfiguration={
        'LambdaFunctionConfigurations': [{
            'LambdaFunctionArn': LAMBDA_ARN,
            'Events': ['s3:ObjectCreated:Put'],
            'Filter': {
                'Key': {
                    'FilterRules': [{
                        'Name': 'suffix',
                        'Value': '.jpg'
                    }]
                }
            }
        }]
    }
)

print(f"✅ S3 trigger added successfully! Bucket: {INPUT_BUCKET}")
print("🎯 Any .jpg file uploaded to the input bucket will now automatically trigger Lambda!")

✅ Lambda invoke permission granted to S3


✅ S3 trigger added successfully! Bucket: rekognition-input-images-745404749079-us-east-1-nb
🎯 Any .jpg file uploaded to the input bucket will now automatically trigger Lambda!


---
## 🖼️ STEP 7: Upload the Image to the Input Bucket
> **First**, upload `retail-shelf.jpg` to the same folder as this notebook using the SageMaker file browser (left sidebar → Upload button).

In [9]:
# Check if the image file exists locally
if not os.path.exists(IMAGE_FILE):
    print(f"❌ '{IMAGE_FILE}' not found! Please upload it to the notebook folder first.")
    print("   In SageMaker: Left sidebar → Upload button → select retail-shelf.jpg")
else:
    s3.upload_file(IMAGE_FILE, INPUT_BUCKET, IMAGE_FILE)
    print(f"✅ Image uploaded to s3://{INPUT_BUCKET}/{IMAGE_FILE}")
    print("⏳ Waiting 15 seconds for Lambda to trigger and process the image...")
    time.sleep(15)
    print("🔁 Now checking the output bucket...")

✅ Image uploaded to s3://rekognition-input-images-745404749079-us-east-1-nb/retail-shelf.jpg
⏳ Waiting 15 seconds for Lambda to trigger and process the image...


🔁 Now checking the output bucket...


---
## 🎵 STEP 8: Check the Output Bucket for the MP3 File

In [10]:
expected_mp3 = IMAGE_FILE.replace('.jpg', '.mp3')

try:
    response = s3.list_objects_v2(Bucket=OUTPUT_BUCKET)
    files = [obj['Key'] for obj in response.get('Contents', [])]

    if files:
        print(f"✅ Files found in the output bucket:")
        for f in files:
            print(f"   🔊 {f}")
    else:
        print("⚠️  Output bucket is still empty. Wait a bit longer or check Lambda logs in Step 11.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Files found in the output bucket:
   🔊 retail-shelf.mp3


---
## ⬇️ STEP 9: Download and Play the MP3 File

In [11]:
from IPython.display import Audio, display

mp3_key   = IMAGE_FILE.replace('.jpg', '.mp3')
local_mp3 = f'/tmp/{mp3_key}'

try:
    s3.download_file(OUTPUT_BUCKET, mp3_key, local_mp3)
    print(f"✅ MP3 downloaded to: {local_mp3}")
    print("🔊 Play it here:")
    display(Audio(local_mp3, autoplay=False))
except Exception as e:
    print(f"❌ MP3 not found yet. Error: {e}")
    print("   → Wait a little longer and run this cell again.")

✅ MP3 downloaded to: /tmp/retail-shelf.mp3
🔊 Play it here:


---
## 🧪 BONUS STEP 10: Direct Pipeline Test (Bypass Lambda — Run Rekognition + Polly Directly)
> Use this cell if the S3 trigger is not working. It calls Rekognition and Polly directly from this notebook — no Lambda needed.

In [13]:
from IPython.display import Audio, display

rekognition  = boto3.client('rekognition', region_name=REGION)
polly_client = boto3.client('polly',       region_name=REGION)

print("🔍 Step 1: Analyzing image with Amazon Rekognition...")
rek_response = rekognition.detect_labels(
    Image={'S3Object': {'Bucket': INPUT_BUCKET, 'Name': IMAGE_FILE}},
    MaxLabels=5,
    MinConfidence=80
)

labels = rek_response['Labels']
print(f"\n📋 Detected Labels ({len(labels)}):")
for label in labels:
    print(f"   🏷️  {label['Name']} — {label['Confidence']:.1f}% confidence")

# Build the spoken description
if labels:
    description = "I can see the following items. " + ", ".join([l['Name'] for l in labels])
else:
    description = "I could not identify any objects in this image."

print(f"\n💬 Generated Description: {description}")

print("\n🗣️  Step 2: Converting text to speech using Amazon Polly...")
polly_response = polly_client.synthesize_speech(
    Text=description,
    OutputFormat='mp3',
    VoiceId='Joanna'
)

# Save MP3 locally
direct_mp3 = '/tmp/direct_output.mp3'
with open(direct_mp3, 'wb') as f:
    f.write(polly_response['AudioStream'].read())

# Also upload MP3 to the output S3 bucket
with open(direct_mp3, 'rb') as f:
    s3.put_object(
        Bucket=OUTPUT_BUCKET,
        Key='direct_output.mp3',
        Body=f.read(),
        ContentType='audio/mpeg'
    )

print(f"\n✅ MP3 saved to: s3://{OUTPUT_BUCKET}/direct_output.mp3")
print("\n🔊 Play it here:")
display(Audio(direct_mp3, autoplay=True))

🔍 Step 1: Analyzing image with Amazon Rekognition...



📋 Detected Labels (5):
   🏷️  Shop — 99.4% confidence
   🏷️  Indoors — 99.2% confidence
   🏷️  Grocery Store — 92.2% confidence
   🏷️  Market — 89.9% confidence
   🏷️  Shelf — 87.5% confidence

💬 Generated Description: I can see the following items. Shop, Indoors, Grocery Store, Market, Shelf

🗣️  Step 2: Converting text to speech using Amazon Polly...



✅ MP3 saved to: s3://rekognition-output-audio-745404749079-us-east-1-nb/direct_output.mp3

🔊 Play it here:


---
## 🔍 STEP 11: Check Lambda Logs for Debugging
> If something is not working, run this cell to view the latest CloudWatch logs from your Lambda function.

In [14]:
logs      = boto3.client('logs', region_name=REGION)
log_group = f'/aws/lambda/{LAMBDA_FUNC}'

try:
    streams = logs.describe_log_streams(
        logGroupName=log_group,
        orderBy='LastEventTime',
        descending=True,
        limit=1
    )

    if streams['logStreams']:
        latest_stream = streams['logStreams'][0]['logStreamName']
        events = logs.get_log_events(
            logGroupName=log_group,
            logStreamName=latest_stream,
            limit=30
        )
        print(f"📋 Latest Lambda Logs ({log_group}):")
        print("-" * 60)
        for event in events['events']:
            print(event['message'].strip())
    else:
        print("⚠️  No logs found yet. Lambda has not been triggered yet.")
except Exception as e:
    print(f"❌ Log group not found: {e}")
    print("   → Lambda may not have been triggered yet.")

📋 Latest Lambda Logs (/aws/lambda/SmartRetailVisionNarrator):
------------------------------------------------------------
INIT_START Runtime Version: python:3.10.mainlinev2.v11	Runtime Version ARN: arn:aws:lambda:us-east-1::runtime:1a99fcab819f9251f632fa77c8eede321a978e610e0adec438cce4b60a01fbf3
START RequestId: 89e57d1a-d925-44f6-b861-36c5af55d514 Version: $LATEST
END RequestId: 89e57d1a-d925-44f6-b861-36c5af55d514
REPORT RequestId: 89e57d1a-d925-44f6-b861-36c5af55d514	Duration: 1409.10 ms	Billed Duration: 1997 ms	Memory Size: 256 MB	Max Memory Used: 92 MB	Init Duration: 587.82 ms


---
## 🧹 STEP 12: Cleanup — Delete All Resources (Avoid Unnecessary Charges)

> ⚠️ **Only run this cell after you have fully completed the lab!**

In [17]:
# ====================================================
# ⚠️  CLEANUP — ALL RESOURCES WILL BE DELETED
# Only run this when you are done with the lab
# ====================================================

CONFIRM_DELETE = True  # ← Change to True when you are ready to delete

if not CONFIRM_DELETE:
    print("⛔ Deletion skipped. Set CONFIRM_DELETE = True in this cell to proceed.")
else:
    print("🗑️  Starting cleanup...")

    # 1. Delete Lambda function
    try:
        lmb.delete_function(FunctionName=LAMBDA_FUNC)
        print(f"✅ Lambda function deleted: {LAMBDA_FUNC}")
    except Exception as e:
        print(f"⚠️  Lambda: {e}")

    # 2. Detach policies and delete IAM role
    try:
        attached = iam.list_attached_role_policies(RoleName=LAMBDA_ROLE)['AttachedPolicies']
        for policy in attached:
            iam.detach_role_policy(RoleName=LAMBDA_ROLE, PolicyArn=policy['PolicyArn'])
        iam.delete_role(RoleName=LAMBDA_ROLE)
        print(f"✅ IAM Role deleted: {LAMBDA_ROLE}")
    except Exception as e:
        print(f"⚠️  IAM Role: {e}")

    # 3. Empty and delete S3 buckets
    for bucket in [INPUT_BUCKET, OUTPUT_BUCKET]:
        try:
            objects = s3.list_objects_v2(Bucket=bucket).get('Contents', [])
            for obj in objects:
                s3.delete_object(Bucket=bucket, Key=obj['Key'])
            s3.delete_bucket(Bucket=bucket)
            print(f"✅ S3 Bucket deleted: {bucket}")
        except Exception as e:
            print(f"⚠️  Bucket {bucket}: {e}")

    print("\n🎉 Cleanup complete! All resources have been deleted.")

🗑️  Starting cleanup...


✅ Lambda function deleted: SmartRetailVisionNarrator


✅ IAM Role deleted: SmartRetailLambdaRole


✅ S3 Bucket deleted: rekognition-input-images-745404749079-us-east-1-nb


✅ S3 Bucket deleted: rekognition-output-audio-745404749079-us-east-1-nb

🎉 Cleanup complete! All resources have been deleted.


---
## ✅ Summary

| Step | What Was Done | AWS Service |
|------|---------------|-------------|
| 1 | Configuration set up | — |
| 2 | Created 2 S3 Buckets | Amazon S3 |
| 3 | Created IAM Role + attached 4 policies | AWS IAM |
| 4 | Applied Bucket Policies | Amazon S3 |
| 5 | Deployed Lambda function | AWS Lambda |
| 6 | Added S3 trigger to Lambda | S3 + Lambda |
| 7 | Uploaded product image | Amazon S3 |
| 8 | Verified MP3 output in bucket | Amazon S3 |
| 9 | Played audio file in the notebook | Amazon Polly |
| 10 | Direct pipeline test (no Lambda) | Rekognition + Polly |
| 11 | Checked Lambda CloudWatch logs | CloudWatch |
| 12 | Cleaned up all resources | All services |

---
**🏆 Congratulations! You have successfully built a Voice-Enabled AI Retail Assistant on AWS!**